# NB06 — Des programmes aux données : explorer Iris
**Introduction à Python • 4A ingénieurs • 2 heures**

Objectif : lire un fichier de mesures, choisir une structure de données, écrire des calculs explicites et représenter les résultats. Nous mobilisons listes, dictionnaires, conditions, boucles et fonctions. Nous introduisons `while`, `csv` et Matplotlib, sans Pandas ni apprentissage automatique.

**Préparation.** Placez `Iris.csv` dans le **même dossier que ce notebook**, puis ouvrez Jupyter depuis ce dossier. Python 3 et Matplotlib doivent être disponibles ; `csv` fait partie de Python. Si l'import de Matplotlib échoue, installez-le dans l'environnement utilisé par le noyau (dans une cellule Jupyter : `%pip install matplotlib`), puis redémarrez le noyau.

**Mode d'emploi.** Exécutez les cellules dans l'ordre avec Maj + Entrée. Les cellules vides vous sont réservées. Prédisez avant d'exécuter ; consultez les aides dans l'ordre puis la correction. Le début est un corrigé projeté : **vous n'avez pas à refaire le challenge du NB05**. Les corrections des exercices suivants sont repliées ; pour les exécuter, copiez leur code dans la cellule de travail correspondante. Des cellules de reprise fournissent ensuite les fonctions utiles à la suite : on peut poursuivre même si un exercice n'est pas terminé.

| Repère | Travail | Durée |
|---|---|---:|
| 0–10 min | Corrigé expliqué du challenge NB05 | 10 min |
| 10–25 min | Répéter avec `while` | 15 min |
| 25–40 min | Comprendre le fichier Iris | 15 min |
| 40–60 min | Lire le CSV et convertir les types | 20 min |
| 60–95 min | Effectifs, moyennes et comparaison | 35 min |
| 95–115 min | Documentation et nuage de points | 20 min |
| 115–120 min | Synthèse | 5 min |

Les temps incluent lecture, échanges, pratique et correction. Les explications servent aussi de support de révision après la séance.

**Repères de lecture V2.** Les nouvelles instructions de lecture de fichier, de vérification et de représentation sont expliquées avant leur utilisation. Ces encadrés servent de référence : pendant les deux heures, suivez la démonstration et consultez le détail au moment où une ligne vous pose question. Les réglages techniques fournis ne sont pas à mémoriser pour réussir les exercices.

## 1. Corrigé projeté du challenge NB05
**Lecture collective uniquement : 2 min de rappel, 6 min de code expliqué, 2 min de vérification.**

Le challenge « Supervision d'une installation » reçoit une liste de dictionnaires. Pour les capteurs **actifs uniquement**, il demande les identifiants en dépassement **strict**, la moyenne des températures, l'identifiant du plus grand dépassement positif et un diagnostic global. Aucun actif donne une moyenne `None` et le diagnostic `"AUCUNE DONNEE ACTIVE"`. Aucun dépassement donne un identifiant maximal `None`. Les données reçues doivent rester intactes.

Nous reprenons les données du NB05 fourni dans `Desktop/python_intro_epf`. Une fonction assure une responsabilité et retourne son résultat ; le programme principal l'affiche. En cas d'égalité entre dépassements maximaux, nous gardons le premier rencontré.

In [ ]:
capteurs = [
    {"id": "T01", "temperature": 42, "seuil": 50, "actif": True},
    {"id": "T02", "temperature": 57, "seuil": 55, "actif": True},
    {"id": "T03", "temperature": 38, "seuil": 45, "actif": False},
    {"id": "T04", "temperature": 63, "seuil": 60, "actif": True},
]

### 1.1 Sélectionner puis identifier
`est_en_alerte` est un **prédicat** : il retourne un booléen. Les deux conditions doivent être vraies : le capteur fonctionne **et** sa température dépasse son seuil. Un capteur au seuil n'est donc pas en alerte.

`capteurs_actifs` construit une liste de dictionnaires. `identifiants_en_alerte` construit une liste de chaînes. Dans chaque cas, la liste commence vide et s'enrichit avec `append`. Le retour est après la boucle, pour visiter tous les capteurs.

In [ ]:
def est_en_alerte(capteur):
    return capteur["actif"] and capteur["temperature"] > capteur["seuil"]

def capteurs_actifs(capteurs):
    resultat = []
    for capteur in capteurs:
        if capteur["actif"]:
            resultat.append(capteur)
    return resultat

def identifiants_en_alerte(capteurs):
    resultat = []
    for capteur in capteurs:
        if est_en_alerte(capteur):
            resultat.append(capteur["id"])
    return resultat

### 1.2 Calculer sans confondre zéro et absence
Pour la moyenne, on sélectionne les actifs avant de calculer. Si cette liste est vide, `return None` empêche la division par zéro. Sinon, `total` contient la somme des températures actives et le dénominateur est **le nombre d'actifs**, pas le nombre total de capteurs.

Pour le maximum, `meilleur_ecart = 0` exclut les écarts négatifs ou nuls. L'identifiant et l'écart sont mis à jour ensemble. Le test strict `>` conserve le premier en cas d'égalité. `None` signifie ici « aucun dépassement positif trouvé » ; cela ne signifie pas « température nulle ».

In [ ]:
def moyenne_active(capteurs):
    actifs = capteurs_actifs(capteurs)
    if len(actifs) == 0:
        return None
    total = 0
    for capteur in actifs:
        total += capteur["temperature"]
    return total / len(actifs)

def plus_grand_depassement(capteurs):
    meilleur_id = None
    meilleur_ecart = 0
    for capteur in capteurs:
        if est_en_alerte(capteur):
            ecart = capteur["temperature"] - capteur["seuil"]
            if ecart > meilleur_ecart:
                meilleur_ecart = ecart
                meilleur_id = capteur["id"]
    return meilleur_id

def diagnostic_global(capteurs):
    if len(capteurs_actifs(capteurs)) == 0:
        return "AUCUNE DONNEE ACTIVE"
    if len(identifiants_en_alerte(capteurs)) > 0:
        return "ALERTE"
    return "NORMAL"

Le diagnostic teste **d'abord la disponibilité des données** : une liste d'alertes vide peut signifier « tout est normal » ou « aucun capteur actif ». La réutilisation des fonctions évite de réécrire leurs règles.

### 1.3 Programme principal et résultats
T01 est actif sans dépassement ; T02 dépasse de 2 °C ; T03 est désactivé ; T04 dépasse de 3 °C. La moyenne est `(42 + 57 + 63) / 3 = 54`. Le capteur le plus chaud et le capteur au plus grand dépassement ne désignent pas nécessairement la même chose : c'est bien l'écart au seuil que nous comparons.

In [ ]:
print("Identifiants en alerte :", identifiants_en_alerte(capteurs))
moyenne = moyenne_active(capteurs)
if moyenne is None:
    print("Moyenne : aucune donnée active")
else:
    print("Moyenne active :", moyenne, "°C")
print("Plus grand dépassement :", plus_grand_depassement(capteurs))
print("Diagnostic :", diagnostic_global(capteurs))

### Lire une vérification avant de l'exécuter
`assert` est un mot-clé de Python : il vérifie que l'expression qui suit est vraie. Par exemple, `assert 2 + 3 == 5` réussit silencieusement. Une condition fausse déclenche `AssertionError` et arrête la cellule : cela invite à examiner le résultat attendu et le résultat obtenu. Ce n'est pas une instruction de calcul à ajouter systématiquement dans vos fonctions.

Dans `assert moyenne_active(capteurs) == 54`, Python appelle d'abord la fonction puis compare son résultat à 54. `==` compare deux valeurs ; `=` affecte une valeur à une variable. Dans `... is None`, on vérifie spécifiquement l'absence de résultat. Enfin, `for cas in [[], desactives]:` parcourt une liste contenant deux jeux de données : la liste vide puis la liste des capteurs désactivés.

### 1.4 Vérification des cas limites
`assert condition` ne produit aucun affichage si la condition est vraie ; sinon il signale une erreur. Ces tests sont fournis : il s'agit de lire ce qu'ils vérifient, sans réécrire le challenge. Le cas au seuil vérifie `>` ; les cas sans actif vérifient l'absence ; le cas d'égalité vérifie notre convention.

In [ ]:
assert identifiants_en_alerte(capteurs) == ["T02", "T04"]
assert moyenne_active(capteurs) == 54
assert plus_grand_depassement(capteurs) == "T04"
assert diagnostic_global(capteurs) == "ALERTE"

desactives = [{"id": "X", "temperature": 99, "seuil": 10, "actif": False}]
au_seuil = [{"id": "S", "temperature": 20, "seuil": 20, "actif": True}]
sans_depassement = [
    {"id": "A", "temperature": 10, "seuil": 20, "actif": True},
    {"id": "B", "temperature": 15, "seuil": 20, "actif": True},
]
for cas in [[], desactives]:
    assert identifiants_en_alerte(cas) == []
    assert moyenne_active(cas) is None
    assert plus_grand_depassement(cas) is None
    assert diagnostic_global(cas) == "AUCUNE DONNEE ACTIVE"
for cas in [au_seuil, sans_depassement]:
    assert identifiants_en_alerte(cas) == []
    assert plus_grand_depassement(cas) is None
    assert diagnostic_global(cas) == "NORMAL"
assert moyenne_active(au_seuil) == 20
assert moyenne_active(sans_depassement) == 12.5
egalite = [
    {"id": "A", "temperature": 12, "seuil": 10, "actif": True},
    {"id": "B", "temperature": 22, "seuil": 20, "actif": True},
]
assert plus_grand_depassement(egalite) == "A"
print("Cas de référence et cas limites vérifiés.")

## 2. Répéter tant qu'une condition est vraie : `while`
**5 min d'explication et de prédiction, 7 min de pratique, 3 min de correction.**

Une boucle `for` visite les éléments d'une collection. Une boucle `while` répète un bloc **tant que sa condition est vraie**. Elle convient lorsque la poursuite dépend d'un état qui évolue.

```python
# Schéma de lecture, pas un programme à exécuter :
# initialiser l'état
# while condition_de_poursuite:
#     faire évoluer l'état
# instruction après la boucle
```

Python teste la condition **avant chaque passage**, exécute le bloc entier si elle est vraie, puis teste à nouveau. Si elle est fausse dès le départ, il n'y a aucun passage. Une évolution doit permettre de rendre la condition fausse ; sinon la boucle peut continuer indéfiniment.

**Prédiction.** Une charge commence à 15 et augmente de 20 jusqu'à atteindre ou dépasser 80. Prévoyez la charge finale et le nombre de passages. Tombe-t-on forcément exactement sur 80 ?

In [ ]:
charge = 15
iterations = 0
while charge < 80:
    charge += 20
    iterations += 1
    print("Après le passage", iterations, ": charge =", charge)
print("État final :", charge, "— passages :", iterations)

La trace donne 35, 55, 75 puis 95 : **4 passages**. À 75, la condition est encore vraie ; le corps entier s'exécute et donne 95. Ce n'est qu'au test suivant que la boucle s'arrête.

| Charge avant le test | `charge < 80` | Charge après le corps |
|---:|:---:|---:|
| 15 | vrai | 35 |
| 35 | vrai | 55 |
| 55 | vrai | 75 |
| 75 | vrai | 95 |
| 95 | faux | aucun passage |

**Diagnostic oral.** Sans `charge += 20`, quelle variable de la condition changerait ? Aucune : la condition resterait vraie. Raisonnez sans lancer cette boucle. Si une boucle ne termine pas, utilisez l'interruption du noyau.

### Exercice — remplir un réservoir
Écrivez `nombre_cycles(volume_initial, ajout, cible)` : elle retourne le nombre de cycles nécessaires pour atteindre ou dépasser la cible. Hypothèse : `ajout > 0`. Le réservoir contient 12 L ; chaque cycle ajoute 7 L ; cible : 50 L. Expliquez l'état initial, la condition de poursuite et les deux mises à jour. Testez aussi un volume initial égal ou supérieur à la cible.

<details><summary>Aide 1 — démarche</summary>

Séparez le volume courant du nombre de cycles. « Il faut encore remplir » signifie que le volume est inférieur à la cible.

</details>

<details><summary>Aide 2 — structure</summary>

Initialisez `volume = volume_initial` et `cycles = 0`. Dans `while volume < cible:`, augmentez le volume puis le compteur. Placez `return` après la boucle.

</details>

<details><summary>Correction — à ouvrir après votre tentative</summary>

```python
def nombre_cycles(volume_initial, ajout, cible):
    # Contrat : ajout strictement positif.
    volume = volume_initial
    cycles = 0
    while volume < cible:
        volume += ajout
        cycles += 1
    return cycles

assert nombre_cycles(12, 7, 50) == 6
assert nombre_cycles(50, 7, 50) == 0
assert nombre_cycles(60, 7, 50) == 0
print(nombre_cycles(12, 7, 50))
```

Après 5 cycles : 47 L, il faut continuer. Après 6 cycles : 54 L, la condition devient fausse. Si le volume atteint déjà la cible au départ, le compteur reste à zéro. Un ajout nul ou négatif ne respecte pas le contrat et pourrait empêcher l’arrêt.

</details>

## 3. Découvrir Iris avant de programmer
**5 min d'observation, 5 min de discussion, 5 min de mise en commun.**

Notre question : **les longueurs et largeurs des pétales diffèrent-elles selon l'espèce dans ce fichier ?** Nous commencerons par compter et calculer des moyennes, puis représenterons les observations individuelles.

Un fichier **CSV** est un fichier texte organisé en lignes et en champs séparés par un délimiteur. Ici, le séparateur est la virgule ; le point est le séparateur décimal. La première ligne contient les noms de colonnes.

Extrait exact du fichier fourni :
```text
Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
1,5.1,3.5,1.4,0.2,Iris-setosa
2,4.9,3.0,1.4,0.2,Iris-setosa
3,4.7,3.2,1.3,0.2,Iris-setosa
```

**Avant de lire la suite :** que représente une ligne ? Quelles colonnes sont des mesures ? Peut-on calculer une moyenne de `Species` ? Pourquoi la moyenne de `Id` ne répondrait-elle pas à une question sur la morphologie ? Proposez une question que ce fichier permet d'examiner.

### Dictionnaire des variables
Une ligne de données représente une fleur mesurée. Un sépale est une pièce externe de la fleur, distincte du pétale. Quatre dimensions sont mesurées en centimètres.

| Colonne exacte | Signification | Type Python visé |
|---|---|---|
| `Id` | Identifiant de l'observation | `int` |
| `SepalLengthCm` | Longueur du sépale, cm | `float` |
| `SepalWidthCm` | Largeur du sépale, cm | `float` |
| `PetalLengthCm` | Longueur du pétale, cm | `float` |
| `PetalWidthCm` | Largeur du pétale, cm | `float` |
| `Species` | Nom de l'espèce | `str` |

`Id` est un identifiant, pas une mesure physique. `Species` est une catégorie : `Iris-setosa`, `Iris-versicolor` ou `Iris-virginica`. Le fichier fourni contient **150 fleurs**, 50 par espèce et aucun champ vide ; nous vérifierons ces propriétés par le code. Ce constat concerne ce fichier, pas tous les fichiers CSV.

**Choix de structure.** Une liste représente l'ensemble des fleurs ; un dictionnaire représente une fleur et associe un nom de colonne à sa valeur. C'est la même organisation que pour les capteurs du NB05. `fleurs[0]` désignera le premier dictionnaire ; `fleurs[0]["PetalLengthCm"]` sa longueur de pétale.

**Prédiction de raisonnement.** Si une espèce a une moyenne plus élevée, cela impose-t-il que chaque fleur de cette espèce soit plus grande que chaque fleur des autres espèces ? Gardez la question ouverte jusqu'au graphique.

### Avant le code : un chemin n'est pas encore un fichier ouvert
Nous avons jusqu'ici écrit les données directement dans Python. Pour accéder à un fichier, il faut d'abord indiquer **où il se trouve**, puis l'ouvrir. Le bloc suivant prépare cet accès ; ses outils ne réalisent encore aucun calcul sur les fleurs.

| Instruction | Comment la lire et pourquoi elle est présente |
|---|---|
| `from pathlib import Path` | Dans le module standard `pathlib`, importer l'outil nommé `Path`. Contrairement à `import csv`, on importe ici directement un nom du module : on pourra écrire `Path(...)`. Aucune installation n'est nécessaire. |
| `chemin = Path("Iris.csv")` | Construire un objet qui représente le chemin vers le fichier et le stocker dans `chemin`. Cela ne lit pas le contenu et ne crée pas le CSV. Le chemin est relatif au dossier de travail du noyau. |
| `Path.cwd()` | Appeler l'outil qui donne le dossier de travail courant (*current working directory*). Les parenthèses demandent son exécution. Son affichage aide à comprendre où Python cherche le fichier. |
| `chemin.is_file()` | Demander à l'objet `chemin` si ce chemin désigne un fichier existant. Le résultat est un booléen. Une fonction attachée à un objet, appelée avec un point, est une **méthode** : vous connaissez déjà cette notation avec `liste.append(...)`. |
| `if not chemin.is_file():` | Exécuter le bloc indenté si le fichier n'existe pas. `not` inverse le booléen : `not False` vaut `True`. |
| `raise FileNotFoundError("...")` | Déclencher volontairement une erreur avec un message utile, plutôt que poursuivre sans fichier. `raise` signifie « signaler une exception » ; `FileNotFoundError` nomme le type d'erreur. Pour cette séance, sachez lire cette protection ; vous n'avez pas à concevoir un système de gestion des erreurs. |

**Prédiction :** si le fichier existe, Python entre-t-il dans le `if` ? Non : `is_file()` donne `True`, donc `not True` donne `False`. La suite peut alors ouvrir le fichier.

## 4. Du fichier texte aux valeurs numériques
**5 min de démonstration, 5 min de types, 7 min de pratique, 3 min de reprise.**

### 4.1 Localiser le fichier
Un chemin relatif comme `Iris.csv` est cherché dans le **dossier de travail du noyau**. Ce dossier correspond habituellement au dossier du notebook, mais peut avoir été changé. La cellule suivante affiche le dossier et vérifie la présence du fichier. Si elle échoue, placez le CSV à côté du notebook et relancez Jupyter depuis ce dossier.

### Déplier la lecture des quatre premières lignes
Le second morceau de la cellule qui suit est une première **lecture textuelle**, avant l'interprétation en colonnes :

```python
with chemin.open(encoding="utf-8-sig") as fichier:
    for numero in range(4):
        print(fichier.readline().rstrip())
```

1. `chemin.open(...)` ouvre le fichier en lecture. Le résultat est un objet permettant de lire ce fichier ; ce n'est ni une liste ni le texte complet.
2. `with ... as fichier:` donne à cet objet le nom `fichier` pendant le bloc indenté. À la sortie du bloc, le fichier est automatiquement fermé, même si une erreur survient. `as` signifie ici « sous le nom de ».
3. `encoding="utf-8-sig"` est un **argument nommé** : le nom à gauche de `=` précise le réglage transmis à `open`. L'encodage indique comment interpréter les octets du fichier comme du texte. UTF-8 convient ici ; la variante `utf-8-sig` accepte aussi un éventuel marqueur UTF-8 au début. C'est un réglage fourni, à ne pas mémoriser pour réaliser les calculs.
4. `range(4)` produit successivement 0, 1, 2, 3 : le corps est exécuté quatre fois. `numero` reçoit ces valeurs mais n'est pas utilisé dans le corps : il sert seulement à répéter la lecture quatre fois.
5. `fichier.readline()` lit **la prochaine ligne**, puis avance la position de lecture. Deux appels successifs lisent donc deux lignes différentes. Le caractère de fin de ligne fait généralement partie de la chaîne renvoyée ; en fin de fichier, le résultat serait `""`.
6. `.rstrip()` enlève les espaces de fin de cette chaîne, notamment le saut de ligne. `print` ajoutant lui-même un saut de ligne, on évite ainsi les lignes blanches entre les affichages. Le fichier sur disque reste intact.

Pour lire l'enchaînement `print(fichier.readline().rstrip())`, partez de l'intérieur : **lire une ligne → nettoyer sa fin pour l'affichage → afficher**. On aurait pu écrire ces trois opérations sur trois lignes. Les quatre lectures affichent l'en-tête et trois fleurs, pas quatre fleurs.

### Comprendre le passage à `DictReader`
Dans la prochaine cellule, on ouvre le fichier à nouveau : la lecture repart donc du début. `open(chemin, ...)` et `chemin.open(...)` sont deux écritures possibles pour ouvrir le même fichier ; nous utilisons ensuite `open` pour suivre les exemples usuels du module `csv`.

- `newline=""` confie la gestion des fins de ligne au module CSV. Cela ne désigne pas le séparateur des colonnes. La virgule est ici le séparateur par défaut de `DictReader`.
- `lecteur = csv.DictReader(fichier)` prépare un lecteur qui transforme les lignes de texte en dictionnaires. Le préfixe `csv.` indique que l'outil vient du module importé avec `import csv`.
- `lecteur.fieldnames` donne les noms de colonnes. Il n'y a pas de parenthèses : on consulte une information de l'objet, appelée **attribut**, plutôt qu'appeler une méthode.
- `for ligne in lecteur:` récupère successivement les dictionnaires des fleurs, sans traiter l'en-tête comme une fleur. `lignes_brutes.append(ligne)` les conserve dans une liste réutilisable.
- `lignes_brutes[0]["PetalLengthCm"]` effectue deux accès : `[0]` sélectionne la première fleur, puis la clé sélectionne un champ de son dictionnaire. `type(...)` permet de vérifier le type de ce champ.

Ces opérations préparent les données. Les boucles et calculs qui suivent réutilisent ensuite les structures déjà connues.

In [ ]:
from pathlib import Path

chemin = Path("Iris.csv")
print("Dossier de travail :", Path.cwd())
if not chemin.is_file():
    raise FileNotFoundError("Placez Iris.csv à côté du notebook et ouvrez Jupyter depuis ce dossier.")

# Lecture textuelle des quatre premières lignes : en-tête et trois fleurs.
with chemin.open(encoding="utf-8-sig") as fichier:
    for numero in range(4):
        print(fichier.readline().rstrip())

### 4.2 Lire avec `csv.DictReader`
`import csv` donne accès à un module de la bibliothèque standard. `with open(...) as fichier:` ouvre le fichier pendant le bloc indenté et le ferme à la sortie. `newline=""` laisse le module CSV gérer les fins de ligne ; `utf-8-sig` accepte aussi un éventuel marqueur UTF-8 en début de fichier.

`csv.DictReader` utilise l'en-tête pour les clés de chaque dictionnaire. Chaque passage de la boucle fournit une ligne de données. Nous l'ajoutons à une liste pour pouvoir parcourir les données plusieurs fois, y compris après la fermeture du fichier. **DictReader lit ici des chaînes, même pour les nombres.**

Prédisez le type de la longueur de pétale avant d'exécuter.

In [ ]:
import csv

lignes_brutes = []
with open(chemin, encoding="utf-8-sig", newline="") as fichier:
    lecteur = csv.DictReader(fichier)
    print("Colonnes :", lecteur.fieldnames)
    for ligne in lecteur:
        lignes_brutes.append(ligne)

print("Nombre de fleurs :", len(lignes_brutes))
print("Première ligne :", lignes_brutes[0])
print("Type lu :", type(lignes_brutes[0]["PetalLengthCm"]))

### 4.3 Convertir : apparence d'un nombre et valeur numérique
`"1.4"` est du texte ; `1.4` est un nombre flottant. La conversion `float("1.4")` permet l'addition numérique. `int("1")` donne un entier. Garder les chaînes produirait soit une erreur lors d'une division, soit une opération différente de celle attendue.

**Prédiction :** quel sera le résultat de chacune des deux additions ?

In [ ]:
print("1.4" + "1.3")       # Concaténation de chaînes
print(float("1.4") + float("1.3"))  # Addition numérique
print(type(int("1")), type(float("1.4")))

### Exercice — convertir une fleur
Écrivez `convertir_fleur(ligne)` qui reçoit un dictionnaire brut et retourne **un nouveau dictionnaire** avec les six clés du fichier : `Id` entier, les quatre mesures flottantes, `Species` chaîne. Ne modifiez pas `ligne`.

Testez sur `lignes_brutes[0]` : l'identifiant doit valoir `1`, la longueur de pétale `1.4`. Vérifiez les types avec `type`. Le fichier fourni est complet ; un champ vide ou une chaîne non numérique nécessiterait une règle de traitement supplémentaire avant conversion.

<details><summary>Aide 1 — démarche</summary>

Recopiez les six clés exactement, majuscules comprises. Appliquez `int` à l’identifiant et `float` à chaque mesure.

</details>

<details><summary>Aide 2 — structure</summary>

Retournez un dictionnaire : `return {"Id": int(ligne["Id"]), ...}`. Pour `Species`, conservez `ligne["Species"]` sans conversion numérique.

</details>

<details><summary>Correction — à ouvrir après votre tentative</summary>

```python
def convertir_fleur(ligne):
    return {
        "Id": int(ligne["Id"]),
        "SepalLengthCm": float(ligne["SepalLengthCm"]),
        "SepalWidthCm": float(ligne["SepalWidthCm"]),
        "PetalLengthCm": float(ligne["PetalLengthCm"]),
        "PetalWidthCm": float(ligne["PetalWidthCm"]),
        "Species": ligne["Species"],
    }

print(convertir_fleur(lignes_brutes[0]))
```

On construit un nouvel objet pour garder la lecture brute disponible. Les clés restent identiques ; les valeurs changent de type. `float` attend ici un point décimal : `float("1,4")` ne conviendrait pas.

</details>

### Lire les contrôles des valeurs brutes
Avant la conversion, le code va vérifier tous les champs :

```python
for ligne in lignes_brutes:
    for valeur in ligne.values():
        assert valeur is not None and valeur.strip() != ""
```

La première boucle choisit une fleur. La seconde parcourt les **valeurs** de son dictionnaire grâce à `.values()` ; elle ne parcourt pas les noms de colonnes. Pour chaque valeur, `is not None` vérifie sa présence et `.strip() != ""` vérifie que le texte n'est pas vide après suppression des espaces au début et à la fin.

`and` évalue d'abord la première condition. Si la valeur est `None`, Python n'essaie pas d'appeler `.strip()` dessus : cette évaluation s'arrête dès que le résultat est déjà faux. Sur le fichier fourni, toutes les vérifications réussissent. Une vérification qui échoue signale une donnée à examiner ; le code ne la corrige pas automatiquement.

La boucle de conversion est distincte : elle appelle `convertir_fleur(ligne)` pour **chaque fleur**, puis ajoute le nouveau dictionnaire à `fleurs`. Enfin, `type(...) is int` vérifie que l'identifiant est bien un entier, pas seulement un texte composé de chiffres.

### Reprise commune — construire et vérifier la liste typée
La définition est redonnée pour que la suite soit exécutable même si la cellule de travail est vide. Comparez-la à votre solution. Nous convertissons chaque ligne puis ajoutons le nouveau dictionnaire à `fleurs`.

La vérification des champs vides porte sur les valeurs textuelles avant conversion. Elle ne prouve pas que les mesures sont exactes : elle vérifie seulement cette propriété du fichier.

In [ ]:
def convertir_fleur(ligne):
    return {
        "Id": int(ligne["Id"]),
        "SepalLengthCm": float(ligne["SepalLengthCm"]),
        "SepalWidthCm": float(ligne["SepalWidthCm"]),
        "PetalLengthCm": float(ligne["PetalLengthCm"]),
        "PetalWidthCm": float(ligne["PetalWidthCm"]),
        "Species": ligne["Species"],
    }

for ligne in lignes_brutes:
    for valeur in ligne.values():
        assert valeur is not None and valeur.strip() != ""

fleurs = []
for ligne in lignes_brutes:
    fleurs.append(convertir_fleur(ligne))

assert len(fleurs) == 150
assert type(fleurs[0]["Id"]) is int
assert type(fleurs[0]["PetalLengthCm"]) is float
assert type(lignes_brutes[0]["PetalLengthCm"]) is str
print(fleurs[0])
print("Type après conversion :", type(fleurs[0]["PetalLengthCm"]))

### Une nouvelle utilisation de `in` : tester la présence d'une clé
Vous connaissez `for element in liste`, qui organise un parcours. Dans `if categorie not in comptages:`, `in` sert à **tester l'appartenance** : pour un dictionnaire, il examine les clés. `not in` signifie « ne figure pas parmi les clés ».

Si `comptages` vaut `{"A": 2}`, alors `"A" in comptages` vaut `True` et `"B" not in comptages` vaut `True`. Il faut créer la clé B avant de lire ou augmenter son compteur, sinon Python signalerait une clé absente (`KeyError`). `comptages[categorie] += 1` signifie `comptages[categorie] = comptages[categorie] + 1`.

## 5. Analyser avec des algorithmes explicites
**10 min pour les effectifs, 15 min pour les moyennes, 10 min pour comparer et interpréter.**

### 5.1 Effectifs : combien de fleurs par espèce ?
Un **effectif** compte des observations. Ici, nous voulons un dictionnaire dont les clés sont les noms d'espèces et les valeurs leurs effectifs. Le même compteur doit être retrouvé et augmenté à chaque nouvelle fleur de cette espèce.

Avant Iris, examinons un exemple réduit. **Prédisez** le dictionnaire final et l'état après le troisième élément.

In [ ]:
categories = ["A", "B", "A", "A"]
comptages = {}
for categorie in categories:
    if categorie not in comptages:
        comptages[categorie] = 0
    comptages[categorie] += 1
    print(categorie, "→", comptages)
print("Résultat :", comptages)

**Lire `fleurs[:3]`.** Cette notation est une **tranche de liste** : elle sélectionne les éléments depuis le début jusqu'à l'indice 3 exclu, donc les indices 0, 1 et 2. Elle produit une nouvelle liste avec ces trois éléments et ne retire rien de `fleurs`. Elle sert ici à vérifier l'algorithme sur un petit cas lisible.

Après le troisième élément : `{"A": 2, "B": 1}`. La première rencontre crée la clé à 0 ; **chaque rencontre**, y compris la première, ajoute 1. L'initialisation ne doit pas être répétée pour une clé déjà présente, sinon son compteur serait remis à zéro.

### Exercice A — un dictionnaire d'effectifs
Écrivez `effectifs_par_espece(donnees)` qui retourne un dictionnaire. Entrée : une liste de fleurs typées. Pour chaque fleur, lisez `"Species"`, initialisez sa clé si nécessaire et incrémentez le compteur. Pour une liste vide, retournez `{}`. Ne codez pas directement « 50 » : votre fonction doit compter les lignes reçues.

Vérifiez d'abord sur `fleurs[:3]` (les trois premières fleurs), puis sur tout le fichier. Pour ces trois premières lignes, l'attente est `{"Iris-setosa": 3}`.

<details><summary>Aide 1 — démarche</summary>

L’état du calcul est un dictionnaire vide au départ. Après chaque passage, il donne les effectifs des fleurs déjà visitées.

</details>

<details><summary>Aide 2 — structure</summary>

Dans `for fleur in donnees:`, posez `espece = fleur["Species"]`. Testez `if espece not in effectifs:` avant la création de sa clé. L’incrément doit être en dehors de ce `if`, mais dans la boucle.

</details>

<details><summary>Correction — à ouvrir après votre tentative</summary>

```python
def effectifs_par_espece(donnees):
    effectifs = {}
    for fleur in donnees:
        espece = fleur["Species"]
        if espece not in effectifs:
            effectifs[espece] = 0
        effectifs[espece] += 1
    return effectifs

assert effectifs_par_espece([]) == {}
assert effectifs_par_espece(fleurs[:3]) == {"Iris-setosa": 3}
print(effectifs_par_espece(fleurs))
```

Le résultat comporte trois clés, chacune associée à 50. Le retour vient après le parcours entier. Un retour placé dans la boucle ne compterait que les premières observations.

</details>

**Comprendre le test numérique fourni dans la correction.** `abs(reel - attendu) < 1e-9` compare l'écart entre deux nombres à une petite tolérance. `abs` donne la valeur absolue : le signe de l'écart n'importe pas. `1e-9` signifie 10 à la puissance −9, soit 0.000000001. Les nombres flottants ne représentent pas exactement tous les décimaux ; on vérifie donc une proximité plutôt qu'une égalité stricte. Ce test s'utilise seulement après avoir établi que le résultat n'est pas `None`.

### 5.2 Moyenne des longueurs de pétales d'une espèce
Une moyenne est une somme divisée par **le nombre de valeurs qui ont contribué à cette somme**. Pour une espèce choisie, il faut donc sélectionner les fleurs de cette espèce dans les deux calculs.

Contrat de `moyenne_petales(donnees, espece)` : retourner un `float` représentant la longueur moyenne des pétales, en cm, ou `None` si aucune fleur de l'espèce demandée n'est présente. La liste et les dictionnaires reçus restent inchangés.

Formule : moyenne de l'espèce = somme de ses longueurs de pétales / effectif de cette espèce.

**Mini-trace à discuter.** Pour `(A, 1.0)`, `(B, 8.0)`, `(A, 3.0)`, si on sélectionne A, les états `(total, compteur)` sont `(1.0, 1)`, `(1.0, 1)`, `(4.0, 2)`. La moyenne est 2.0. Diviser par 3 ferait entrer une fleur exclue dans le dénominateur.

### Exercice B — sélectionner et agréger
Implémentez cette fonction avec une boucle, un accumulateur `total` et un compteur. Placez les deux mises à jour sous le **même test d'espèce**. Traitez l'absence avant de diviser. Essayez `"Iris-setosa"`, puis `"Espèce absente"` et une liste vide.

<details><summary>Aide 1 — démarche</summary>

Initialisez `total = 0.0` et `compteur = 0`. Ne mettez à jour ces variables que si l’espèce de la fleur correspond au paramètre.

</details>

<details><summary>Aide 2 — structure</summary>

Après la boucle, si `compteur == 0`, retournez `None`. Sinon retournez `total / compteur`. La division n’a pas sa place à chaque passage.

</details>

<details><summary>Correction — à ouvrir après votre tentative</summary>

```python
def moyenne_petales(donnees, espece):
    total = 0.0
    compteur = 0
    for fleur in donnees:
        if fleur["Species"] == espece:
            total += fleur["PetalLengthCm"]
            compteur += 1
    if compteur == 0:
        return None
    return total / compteur

assert moyenne_petales([], "Iris-setosa") is None
assert moyenne_petales(fleurs, "Espèce absente") is None
assert abs(moyenne_petales(fleurs, "Iris-setosa") - 1.464) < 1e-9
print(moyenne_petales(fleurs, "Iris-setosa"))
```

`total` est une somme de longueurs, `compteur` un nombre de fleurs. `None` indique une moyenne impossible faute de données ; une moyenne de zéro serait un résultat numérique différent. La tolérance du test évite de confondre l’arrondi des flottants et une erreur de calcul.

</details>

### Lire le dictionnaire de résultats
Dans `for espece in effectifs:`, le parcours du dictionnaire donne ses **clés**, ici les trois noms d'espèces. À chaque passage, `moyennes[espece] = moyenne_petales(fleurs, espece)` calcule une valeur pour ce nom et la stocke sous cette même clé dans un autre dictionnaire.

`round(moyennes[espece], 3)` appelle une fonction avec deux arguments : le nombre à arrondir puis le nombre de décimales demandé. Cela ne modifie pas la valeur stockée. L'affichage peut donner `4.26` plutôt que `4.260` : le nombre est le même, car `round` ne fixe pas une présentation avec des zéros finaux.

### Reprise commune — données de comparaison
Les deux fonctions sont fournies ci-dessous pour assurer la continuité du travail. On calcule ensuite un dictionnaire de moyennes. Parcourir un dictionnaire avec `for espece in effectifs:` visite ses clés.

`round(valeur, 3)` arrondit ici pour l'affichage ; les valeurs stockées dans `moyennes` restent les résultats non arrondis.

In [ ]:
def effectifs_par_espece(donnees):
    effectifs = {}
    for fleur in donnees:
        espece = fleur["Species"]
        if espece not in effectifs:
            effectifs[espece] = 0
        effectifs[espece] += 1
    return effectifs

def moyenne_petales(donnees, espece):
    total = 0.0
    compteur = 0
    for fleur in donnees:
        if fleur["Species"] == espece:
            total += fleur["PetalLengthCm"]
            compteur += 1
    if compteur == 0:
        return None
    return total / compteur

effectifs = effectifs_par_espece(fleurs)
moyennes = {}
for espece in effectifs:
    moyennes[espece] = moyenne_petales(fleurs, espece)
    print(espece, "— effectif :", effectifs[espece],
          "— moyenne des longueurs de pétales :", round(moyennes[espece], 3), "cm")

assert effectifs == {"Iris-setosa": 50, "Iris-versicolor": 50, "Iris-virginica": 50}
assert abs(moyennes["Iris-versicolor"] - 4.260) < 1e-9
assert abs(moyennes["Iris-virginica"] - 5.552) < 1e-9

**Lire le test de première rencontre dans la correction.** L'expression `meilleure_moyenne is None or valeur > meilleure_moyenne` se lit « aucun candidat encore retenu, ou nouvelle valeur plus grande ». `or` évalue la partie gauche d'abord. Si elle est vraie, il n'évalue pas la comparaison de droite : on évite ainsi de comparer un nombre à `None`. Une fois un nombre retenu, la comparaison de droite sert à rechercher un meilleur candidat. Le test strict conserve le premier en cas d'égalité.

### Exercice C — comparer et argumenter
1. À partir de `moyennes`, trouvez par une boucle l'espèce ayant la moyenne la plus élevée. Initialisez l'espèce retenue à `None` et la meilleure moyenne à `None`. Ignorez une moyenne absente. Conservez la première en cas d'égalité.
2. Calculez l'écart de moyenne entre `Iris-virginica` et `Iris-versicolor`.
3. Rédigez deux phrases : une observation chiffrée sur ce fichier, puis une limite de cette observation. Peut-on conclure sur **chaque fleur** en comparant seulement les moyennes ?

Les trois moyennes de référence sont 1.464, 4.260 et 5.552 cm. Utilisez les valeurs calculées pour rechercher le maximum ; n'écrivez pas le nom gagnant directement dans votre algorithme.

**Votre interprétation (double-cliquez pour compléter cette cellule Markdown)**

<details><summary>Aide 1 — démarche</summary>

Comme dans le challenge des capteurs, mémorisez ensemble la valeur maximale et le nom associé.

</details>

<details><summary>Aide 2 — structure</summary>

Si `valeur is not None`, retenez-la quand `meilleure_moyenne is None` ou quand elle dépasse `meilleure_moyenne`. Le test avec `or` permet de traiter la première valeur sans comparer un nombre à `None`.

</details>

<details><summary>Correction — à ouvrir après votre tentative</summary>

```python
meilleure_espece = None
meilleure_moyenne = None
for espece in moyennes:
    valeur = moyennes[espece]
    if valeur is not None:
        if meilleure_moyenne is None or valeur > meilleure_moyenne:
            meilleure_espece = espece
            meilleure_moyenne = valeur
print("Moyenne maximale :", meilleure_espece, meilleure_moyenne, "cm")
ecart = moyennes["Iris-virginica"] - moyennes["Iris-versicolor"]
print("Écart virginica − versicolor :", round(ecart, 3), "cm")
```

Dans ce fichier, `Iris-virginica` présente la moyenne la plus élevée : 5.552 cm, soit 1.292 cm de plus que `Iris-versicolor`. Cette comparaison résume les groupes ; elle ne décrit pas leur dispersion et ne prouve pas que toute fleur virginica dépasse toute fleur versicolor. Ces données seules n’établissent pas une explication causale ni une généralisation à toutes les fleurs.

</details>

### Avant l'exemple : décoder les commandes graphiques
La bibliothèque doit recevoir des données et des consignes de présentation. Les noms suivants sont nouveaux ; voici précisément ce qu'ils demandent :

| Ligne ou argument | Rôle dans la construction de la figure |
|---|---|
| `import matplotlib.pyplot as plt` | Importer le module graphique `pyplot` en lui donnant l'alias court `plt`. `plt` n'est pas un mot-clé : c'est le nom choisi pour appeler ses outils. |
| `plt.figure(figsize=(6, 4))` | Créer une nouvelle figure. `figsize` règle sa largeur et sa hauteur en pouces, pas les limites des axes. `(6, 4)` est un couple de valeurs, appelé tuple, transmis comme un seul argument. |
| `plt.scatter(x, y, ...)` | Ajouter une série de points à la figure courante. Le premier point associe `x[0]` à `y[0]`, le deuxième `x[1]` à `y[1]`, etc. Les deux listes doivent avoir la même longueur. |
| `label="Exemple fictif"` | Associer un nom à cette série pour la légende. Ce texte n'est ni une variable à rechercher ni une nouvelle donnée numérique. |
| `alpha=0.7` | Régler l'opacité de la série : les points sont partiellement transparents, ce qui aide à voir les superpositions. |
| `plt.xlabel(...)`, `plt.ylabel(...)` | Nommer respectivement l'axe horizontal et l'axe vertical. Les unités doivent être indiquées. |
| `plt.title(...)`, `plt.legend()` | Ajouter un titre et afficher les noms des séries renseignés par `label`. |
| `plt.tight_layout()` | Ajuster les marges pour laisser de la place aux titres et aux noms d'axes. Cela ne change pas les données. |
| `plt.show()` | Demander l'affichage de la figure construite. |

Dans `label="Exemple fictif"`, le signe `=` sert à nommer un **argument de l'appel**, comme `encoding=` plus haut. Il ne crée pas une variable `label` dans votre notebook. Les listes de l'exemple sont des données fictives pour comprendre l'outil ; nous les remplacerons ensuite par les mesures du CSV.

## 6. Découvrir Matplotlib à partir de sa documentation
**5 min de lecture documentaire, 10 min de construction, 5 min d'interprétation.**

Une bibliothèque fournit des outils déjà programmés. Nous choisissons **Matplotlib** pour représenter les données. Un nuage de points associe à chaque fleur un couple `(longueur du pétale, largeur du pétale)` ; la couleur et la forme identifient son espèce.

### 6.1 Chercher une information précise
Ouvrez la [documentation officielle de `plt.scatter`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html). Repérez les paramètres `x`, `y`, `label`, `alpha` et `marker`. Quelle information devez-vous préparer pour `x` et `y` ? À quoi sert `alpha` lorsque des points se superposent ? `label` donne un nom à la série, mais il faut ensuite demander l'affichage de la légende.

**Repère autonome si vous n'avez pas accès au site :** `plt.scatter(x, y, label="nom", alpha=0.7, marker="o")` trace une série. `x` et `y` contiennent autant de valeurs, dans le même ordre : les valeurs au même indice décrivent le même point. `alpha` va de 0 (transparent) à 1 (opaque). `marker` choisit la forme. Les paramètres nommés explicitent le rôle de chaque argument.

### 6.2 Un exemple minimal exécutable
Prédisez les coordonnées des trois points. `import matplotlib.pyplot as plt` donne au module le nom court `plt`. `figure` crée la figure ; `xlabel` et `ylabel` nomment les axes ; `legend` affiche les noms des séries ; `show` affiche le résultat.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.scatter([1.0, 2.0, 3.0], [0.2, 0.4, 0.5], label="Exemple fictif", alpha=0.7)
plt.xlabel("Longueur (cm)")
plt.ylabel("Largeur (cm)")
plt.title("Trois points pour comprendre scatter")
plt.legend()
plt.tight_layout()
plt.show()

### Préparer le graphique Iris : deux listes qui doivent rester associées
Pour une espèce, `longueurs` reçoit les abscisses et `largeurs` les ordonnées. Si une fleur a un pétale de 1.4 cm de long et 0.2 cm de large, il faut ajouter ces deux nombres au même passage : ils décrivent **une seule fleur**. On vide les deux listes pour chaque nouvelle espèce, mais on crée la figure une seule fois pour conserver les trois séries ensemble.

La correction utilise aussi `marqueurs`, un dictionnaire qui associe une espèce à un symbole : `"o"` pour un cercle, `"s"` pour un carré et `"^"` pour un triangle. `marker=marqueurs[espece]` transmet à `scatter` le symbole choisi pour l'espèce courante ; il ne s'agit pas d'un calcul sur les fleurs. Matplotlib choisit automatiquement une couleur pour chaque série.

Deux réglages de présentation complètent la correction : `plt.legend(title="Espèce")` donne un titre à la légende ; `plt.grid(alpha=0.2)` ajoute une grille légère. `plt.figure(figsize=(8, 5))` crée une figure plus large que dans l'exemple minimal. L'instruction `assert len(longueurs) == len(largeurs) == 50` vérifie que chacune des deux listes contient 50 valeurs ; ce nombre concerne le fichier fourni.

### Exercice D — représenter les trois espèces
Construisez une figure avec **une série par espèce**. Pour chaque espèce, créez deux listes vides, parcourez `fleurs` et ajoutez la longueur et la largeur du pétale de chaque fleur retenue. Ajoutez les deux valeurs **sous le même test**, dans la même boucle : elles doivent rester appariées.

Appelez ensuite `plt.scatter` avec ces listes et un `label`. Donnez aux axes leurs noms et unités, ajoutez titre et légende puis affichez. Vous pouvez utiliser `especes = ["Iris-setosa", "Iris-versicolor", "Iris-virginica"]`.

**Avant de lancer :** chaque série doit recevoir 50 couples ; au total 150 fleurs seront représentées. Il peut y avoir moins de points visuellement distincts parce que des mesures sont identiques.

<details><summary>Aide 1 — démarche</summary>

La création de la figure se place avant la boucle sur les espèces. Les deux listes se réinitialisent à l’intérieur de cette boucle, avant le parcours des fleurs.

</details>

<details><summary>Aide 2 — structure</summary>

Pour une fleur de l’espèce retenue : `longueurs.append(fleur["PetalLengthCm"])` et `largeurs.append(fleur["PetalWidthCm"])`. L’appel à `scatter` vient après ce parcours, une fois les deux listes complètes.

</details>

<details><summary>Correction — à ouvrir après votre tentative</summary>

```python
especes = ["Iris-setosa", "Iris-versicolor", "Iris-virginica"]
marqueurs = {"Iris-setosa": "o", "Iris-versicolor": "s", "Iris-virginica": "^"}
plt.figure(figsize=(8, 5))
for espece in especes:
    longueurs = []
    largeurs = []
    for fleur in fleurs:
        if fleur["Species"] == espece:
            longueurs.append(fleur["PetalLengthCm"])
            largeurs.append(fleur["PetalWidthCm"])
    assert len(longueurs) == len(largeurs) == 50
    plt.scatter(longueurs, largeurs, label=espece,
                marker=marqueurs[espece], alpha=0.7)
plt.xlabel("Longueur du pétale (cm)")
plt.ylabel("Largeur du pétale (cm)")
plt.title("Iris — dimensions des pétales par espèce")
plt.legend(title="Espèce")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()
```

Une série correspond à une espèce, les deux listes à deux mesures des mêmes fleurs. Les formes différentes complètent les couleurs. Les appels de mise en forme et `show` sont après la boucle : ils portent sur la figure entière. Ne triez jamais indépendamment les deux listes : vous détruiriez l’association longueur/largeur.

</details>

### Reprise commune — graphique de référence
Exécutez cette cellule après votre tentative pour comparer la représentation. Elle utilise les données déjà chargées et reste exécutable si votre cellule d'exercice est vide.

In [ ]:
especes = ["Iris-setosa", "Iris-versicolor", "Iris-virginica"]
marqueurs = {"Iris-setosa": "o", "Iris-versicolor": "s", "Iris-virginica": "^"}
plt.figure(figsize=(8, 5))
for espece in especes:
    longueurs = []
    largeurs = []
    for fleur in fleurs:
        if fleur["Species"] == espece:
            longueurs.append(fleur["PetalLengthCm"])
            largeurs.append(fleur["PetalWidthCm"])
    assert len(longueurs) == len(largeurs) == 50
    plt.scatter(longueurs, largeurs, label=espece,
                marker=marqueurs[espece], alpha=0.7)
plt.xlabel("Longueur du pétale (cm)")
plt.ylabel("Largeur du pétale (cm)")
plt.title("Iris — dimensions des pétales par espèce")
plt.legend(title="Espèce")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### Interpréter sans surinterpréter
Décrivez les positions et les recouvrements, puis revenez à la question sur les moyennes :

- Où se situent les fleurs setosa ?
- Les deux autres espèces occupent-elles des zones entièrement séparées ?
- Une même longueur de pétale peut-elle s'observer dans plusieurs espèces ?
- Pourquoi ce graphique apporte-t-il une information que les trois moyennes ne donnaient pas ?

<details><summary>Éléments de lecture — après observation</summary>

Dans ce fichier, setosa forme un groupe de pétales courts et étroits, nettement à l'écart des deux autres espèces sur ces axes. Versicolor et virginica ont des zones de recouvrement : leurs moyennes différentes ne rendent pas toutes les observations séparées. Le graphique montre l'étendue et la dispersion des mesures ainsi que leur variation conjointe. Il ne prouve pas qu'une dimension cause l'autre et ne suffit pas à conclure sur une population plus large.

</details>

## 7. Synthèse
**2 min de reformulation individuelle, 3 min de mise en commun.**

Le chemin suivi est : **fichier texte → liste de dictionnaires bruts → conversion des types → sélection → calcul → représentation → interprétation**.

| Besoin | Outil et point d'attention |
|---|---|
| Répéter jusqu'à un état cible | `while` ; justifier ce qui provoque l'arrêt |
| Lire des lignes avec un en-tête | `csv.DictReader` ; les nombres sont d'abord du texte |
| Préparer un calcul | `int`, `float` ; conserver les catégories comme chaînes |
| Compter par catégorie | dictionnaire de compteurs ; créer une clé une seule fois |
| Calculer une moyenne de groupe | même sélection pour somme et dénominateur |
| Traiter l'absence | `None` avant toute division ou comparaison numérique |
| Comparer des observations | nuage de points, axes avec unités, légende |
| Vérifier | prédiction, cas simples, cas limites et valeurs de référence |

**Billet de sortie.** Expliquez en quelques phrases : pourquoi convertir avant de calculer ; pourquoi diviser par 150 ne donne pas la moyenne d'une espèce ; ce qui assure l'arrêt d'une boucle `while` ; ce que le graphique révèle au-delà des moyennes.

**À retenir pour travailler seul.** Lisez les types et la structure avant d'écrire une boucle. Définissez ce que vos variables représentent, puis vérifiez qu'elles gardent ce sens après chaque passage. Une figure se lit avec ses unités et ses limites.

**Références pour réviser :** [module `csv`, documentation Python](https://docs.python.org/fr/3/library/csv.html) ; [nuage de points `scatter`, documentation Matplotlib](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html). Les données de travail sont le fichier `Iris.csv` fourni avec cette séance. Le corrigé initial reprend le challenge du NB05 désigné dans `Desktop/python_intro_epf`, sans modifier ce document.